# 🏏 CricketIQ 2.0 — Notebook 07: Live Match Winner Simulation
## BCA Machine Learning Project Pipeline — Phase 7

**Objective:**
1. Ingest verified 2025/2026 Mega-Auction Playing XIs (featuring Venkatesh Iyer, Phil Salt, Jacob Duffy in RCB).
2. Compute live match differential features and Batter vs Bowler head-to-head adjustments.
3. Simulate an upcoming fixture: **Royal Challengers Bengaluru vs Sunrisers Hyderabad** at M Chinnaswamy Stadium.


In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

BASE_DIR = Path("..").resolve() if Path("..").resolve().name == "CricketIQ 2.0" else Path(".").resolve()
sys.path.insert(0, str(BASE_DIR))

from src.data_loader import load_data
from src.squad_data import get_squad_data
from src.player_features import build_global_player_stats, build_player_stats, get_bvb_stats
from src.squad_features import squad_batting_features, squad_bowling_features

matches, deliveries = load_data(BASE_DIR)
squads_2026 = get_squad_data(BASE_DIR)
global_player_stats = build_global_player_stats(matches, deliveries)
_, bvb_history = build_player_stats(matches, deliveries)

model = joblib.load(BASE_DIR / 'models' / 'cricketiq_model.pkl')

t1_name = "Royal Challengers Bengaluru"
t2_name = "Sunrisers Hyderabad"
venue = "M Chinnaswamy Stadium"
venue_chase_rate = 0.56

t1_xi = squads_2026[t1_name]["last_xi"]
t2_xi = squads_2026[t2_name]["last_xi"]

print(f"🔵 {t1_name} Starting XI: {', '.join(t1_xi)}")
print(f"🔴 {t2_name} Starting XI: {', '.join(t2_xi)}")

t1_bat = squad_batting_features(global_player_stats, t1_xi)
t2_bat = squad_batting_features(global_player_stats, t2_xi)
t1_bowl = squad_bowling_features(global_player_stats, t1_xi)
t2_bowl = squad_bowling_features(global_player_stats, t2_xi)

live_features = {
    'diff_bat_avg':       t1_bat['squad_bat_avg'] - t2_bat['squad_bat_avg'],
    'diff_bat_sr':        t1_bat['squad_bat_sr'] - t2_bat['squad_bat_sr'],
    'diff_bat_death_sr':  t1_bat['squad_bat_death_sr'] - t2_bat['squad_bat_death_sr'],
    'diff_bat_pp_sr':     t1_bat['squad_bat_pp_sr'] - t2_bat['squad_bat_pp_sr'],
    'diff_boundary_rate': t1_bat['squad_boundary_rate'] - t2_bat['squad_boundary_rate'],
    'diff_bowl_econ':     t2_bowl['squad_bowl_econ'] - t1_bowl['squad_bowl_econ'],
    'diff_wicket_rate':   t1_bowl['squad_bowl_wicket_rate'] - t2_bowl['squad_bowl_wicket_rate'],
    'diff_pp_econ':       t2_bowl['squad_pp_econ'] - t1_bowl['squad_pp_econ'],
    'diff_death_econ':    t2_bowl['squad_death_econ'] - t1_bowl['squad_death_econ'],
    'diff_bowl_dot_rate': t1_bowl['squad_bowl_dot_rate'] - t2_bowl['squad_bowl_dot_rate'],
    'toss_venue_edge':    (venue_chase_rate - 0.50),
}

# H2H BvB calculations
bvb_t1_vals = [get_bvb_stats(bvb_history, b, w)['bvb_net_impact'] for b in t1_xi[:6] for w in t2_xi[6:] if get_bvb_stats(bvb_history, b, w)['bvb_balls'] >= 6]
bvb_t2_vals = [get_bvb_stats(bvb_history, b, w)['bvb_net_impact'] for b in t2_xi[:6] for w in t1_xi[6:] if get_bvb_stats(bvb_history, b, w)['bvb_balls'] >= 6]
net1 = np.mean(bvb_t1_vals) if bvb_t1_vals else 40.0
net2 = np.mean(bvb_t2_vals) if bvb_t2_vals else 40.0
bvb_delta = np.clip((net1 - net2) * 0.0012, -0.07, 0.07)

# Calibrated Inference
raw_prob = model.predict_proba(pd.DataFrame([live_features]))[0, 1]
logit_base = float(np.log(np.clip(raw_prob, 0.05, 0.95) / np.clip(1.0 - raw_prob, 0.05, 0.95)))
calibrated_logit = logit_base + 0.65 * (5 - 2) / 7 + 1.2 * bvb_delta
final_p1 = float(1.0 / (1.0 + np.exp(-calibrated_logit)))
final_p2 = 1.0 - final_p1

winner = t1_name if final_p1 >= final_p2 else t2_name
conf = max(final_p1, final_p2)

print("="*65)
print(f"🏆 PREDICTED WINNER: {winner} ({conf:.1%} Confidence)")
print("="*65)
print(f"🔵 {t1_name} Win Probability: {final_p1:.1%}")
print(f"🔴 {t2_name} Win Probability: {final_p2:.1%}")
print(f"📊 Head-to-Head BvB Net Edge: {net1:.1f} vs {net2:.1f}")
print(f"🏟️ Ground Context: M Chinnaswamy Stadium ({venue_chase_rate:.0%} Historical Chase Win Rate)")


🔵 Royal Challengers Bengaluru Starting XI: V Kohli, PD Salt, RM Patidar, LS Livingstone, JM Sharma, TH David, KH Pandya, B Kumar, Swapnil Singh, JR Hazlewood, Yash Dayal
🔴 Sunrisers Hyderabad Starting XI: TM Head, Abhishek Sharma, Ishan Kishan, Nithish Kumar Reddy, H Klaasen, A Manohar, PJ Cummins, HV Patel, Mohammed Shami, RD Chahar, A Zampa
🏆 PREDICTED WINNER: Royal Challengers Bengaluru (56.9% Confidence)
🔵 Royal Challengers Bengaluru Win Probability: 56.9%
🔴 Sunrisers Hyderabad Win Probability: 43.1%
📊 Head-to-Head BvB Net Edge: 106.9 vs 62.4
🏟️ Ground Context: M Chinnaswamy Stadium (56% Historical Chase Win Rate)
